# Route a case question to one specialist

This lab uses a small router agent to choose either the network or evidence specialist. The router selects the route; it does not investigate the case itself.

![Router-supervisor workflow](figures/router-supervisor-workflow.svg)

For this example question, the router selects the network path. The original case and question are then sent only to the network specialist.

## Step 1: Import AgentScope and configuration helpers

This cell imports the agent, message, model, and local-configuration classes used for routing and specialist responses.

In [ ]:
import os

from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel


## Step 2: Configure the shared model connection

The router and both specialists use this connection. Each agent still has its own instructions and conversation state.

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=120),
)


## Step 3: Create the router and two specialists

The router returns a route label only. The specialists perform the actual case review after Python selects one of them.

In [ ]:
router = Agent(
    name="case_router",
    system_prompt=(
        "You route practice security questions. Reply with exactly NETWORK or EVIDENCE. "
        "Choose NETWORK for questions about IP addresses, connections, timestamps, or alert observations. "
        "Choose EVIDENCE for questions about supported facts, missing evidence, or next evidence to collect."
    ),
    model=model,
    react_config=ReActConfig(max_iters=1),
)

network_specialist = Agent(
    name="network_specialist",
    system_prompt=(
        "You are the network specialist for a practice security case. "
        "Report what the network alert observed and what the alert cannot show."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)

evidence_specialist = Agent(
    name="evidence_specialist",
    system_prompt=(
        "You are the evidence specialist for a practice security case. "
        "List supported facts, missing evidence, and one next item to collect."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)


## Step 4: Define the practice case and analyst question

The router receives the question because it needs to choose expertise. The selected specialist receives both the question and these original case materials.

In [ ]:
case_materials = """
Practice case INC-204
- Network-monitoring alert: at 09:14 UTC, an automated sensor recorded a workstation making forty-three outbound contacts to 192.0.2.44.
- The local practice list marks 192.0.2.44 as suspicious and says this requires analyst review; it is not proof of malicious activity.
- The alert does not identify the process, user action, payload, or destination port/service for the contacts.
""".strip()

analyst_question = "What did the alert observe about the connections to 192.0.2.44?"
print(analyst_question)


## Step 5: Ask the router for a route label

This helper extracts text from an AgentScope response. The router sees only the analyst question and should return one of the two short labels.

In [ ]:
def response_text(response: Msg) -> str:
    """Extract readable text from an AgentScope response message."""
    return "".join(block.text for block in response.content if isinstance(block, TextBlock))

router_request = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=analyst_question)],
)

router_response = await router.reply(router_request)
route_label = response_text(router_response).strip().upper()
print(f"Router chose: {route_label}")


## Step 6: Validate the route and choose a specialist

Python checks the router’s text before dispatching. An unexpected reply produces a clear error instead of silently choosing a path.

In [ ]:
specialists = {
    "NETWORK": network_specialist,
    "EVIDENCE": evidence_specialist,
}

if route_label not in specialists:
    raise RuntimeError(
        f"Unexpected route {route_label!r}. The router must return NETWORK or EVIDENCE."
    )

selected_specialist = specialists[route_label]
print(f"Selected specialist: {selected_specialist.name}")


## Step 7: Send the case to the selected specialist

Only the chosen agent receives this request. The output shows both the selected route and that specialist’s response.

In [ ]:
specialist_request = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=(
        f"Practice case:\n{case_materials}\n\n"
        f"Analyst question: {analyst_question}"
    ))],
)

specialist_response = await selected_specialist.reply(specialist_request)
print(f"{route_label} SPECIALIST RESPONSE:")
print(response_text(specialist_response))


## Step 8: Checkpoint

Change `analyst_question` to ask what evidence should be collected before reaching a conclusion. Rerun Steps 5–7 and confirm that the router chooses `EVIDENCE`.